## Loading Vectors into QDrant

In [ ]:
# NOTE: Run the following command in the terminal to pull QDrant database image
# to the Docker container.
#
# Command: docker pull qdrant/qdrant
#
# As a result of the above command, the QDrant image will be pulled to the Docker
# container. We can verify this by running the following command in the terminal.
#
# Command: docker images
#
# Also, we can check the same by navigating to the "Images" section in the Docker
# Desktop application.
# Next, run the following command to run the QDrant image and map port number 6333
# of QDrant to port 6333 of localhost.
#
# Command: docker run -p 6333:6333 qdrant/qdrant
#
# As a result of the above command, the QDrant image will be run in the Docker
# container.
# We can verify this by navigating to the "Containers" section in the Docker
# Desktop application.

### Installing Libraries

In [ ]:
%pip install openai pypdf qdrant-client

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
claude_model = os.getenv("CLAUDE_MODEL")
openai_api_key = os.getenv("OPENAI_API_KEY")

### Creating the OpenAI Client

In [ ]:
from openai import OpenAI

openai_client = OpenAI(api_key=openai_api_key)

### Preparing the Dataset

In [ ]:
from pathlib import Path
from pypdf import PdfReader
import json

pre_vector_data = []

data_dir = Path("Data")

for file in data_dir.glob("*.pdf"):

    reader = PdfReader(file)

    content = ""

    for page in reader.pages:
        content += (page.extract_text() or "") + "\n"

    pre_vector_data.append({
        "title": file.stem,
        "content": content.strip()
    })

# Write to JSON
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(pre_vector_data, f, indent=4, ensure_ascii=False)

print(f"Saved {len(pre_vector_data)} documents to data.json")

### Generating Vector Embedding from Dataset

In [ ]:
with open("data.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

for document in documents:

    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=document["content"]
    )

    document["vector"] = response.data[0].embedding

with open("data.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, indent=4, ensure_ascii=False)

print(f"Generated embeddings for {len(documents)} documents.")

### Uploading Vectors to QDrant Database

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "Travel_guide_for_India"

qdrant_client = QdrantClient(url=QDRANT_URL)


if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=1536, distance=Distance.DOT)
    )
    print(f"Collection '{COLLECTION_NAME}' created.")

    # Load vectors from file
    with open("data.json", "r", encoding='utf-8') as f:
        data = json.load(f)

    points = [
        PointStruct(
            id=i + 1,
            vector=entry["vector"],
            payload={"content": entry["content"]}
        ) for i, entry in enumerate(data)
    ]

    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        wait=True,
        points=points
    )
    print(f"Upserted {len(points)} vectors.")
else:
    print(f"Collection '{COLLECTION_NAME}' already exists.")